# 03 — Görev 3: Modelleme

3 farklı algoritma, hepsi `GridSearchCV` + 5-fold StratifiedCV ile:
- **Logistic Regression** — lineer baseline, yorumlanabilir
- **Random Forest** — ağaç ensemble, non-lineer ilişkileri yakalar
- **XGBoost** — gradient boosting, genelde en iyi performans

Çıktılar:
- 3 model × 2 set (train/test) = 6 confusion matrix
- İlk metrik tablosu (train / CV-mean / test) — CSV + MD
- Eğitilmiş modeller `models/` altına kaydedilir


In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.append(str(Path.cwd() / "src"))
from src import config as C
from src import utils as U

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
np.random.seed(C.RANDOM_STATE)

## 1. Veriyi ve seçilen özellikleri yükle

In [2]:
X_train = pd.read_csv(C.PROC_DIR / "X_train_scaled.csv")
X_test = pd.read_csv(C.PROC_DIR / "X_test_scaled.csv")
y_train = pd.read_csv(C.PROC_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(C.PROC_DIR / "y_test.csv").squeeze("columns")

selected = joblib.load(C.MODELS_DIR / "selected_features.joblib")
X_train = X_train[selected]
X_test = X_test[selected]

print(f"Seçili özellik sayısı: {len(selected)}")
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Seçili özellik sayısı: 7
Train: (227, 7)  |  Test: (61, 7)


## 2. Modelleri ve param-grid'leri tanımla
Küçük veri seti → küçük grid'ler. 5-fold CV → her grid noktası 5 kez eğitilir.

In [3]:
cv = StratifiedKFold(n_splits=C.CV_FOLDS, shuffle=True, random_state=C.RANDOM_STATE)

# scale_pos_weight = neg/pos oranı (XGBoost için dengesizlik düzeltmesi)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "LogisticRegression": (
        LogisticRegression(max_iter=2000, random_state=C.RANDOM_STATE,
                           class_weight="balanced"),
        {
            "C": [0.01, 0.1, 1.0, 10.0],
            "penalty": ["l2"],
            "solver": ["lbfgs"],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=C.RANDOM_STATE,
                               class_weight="balanced", n_jobs=-1),
        {
            "n_estimators": [200, 400],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5],
        },
    ),
    "XGBoost": (
        XGBClassifier(
            random_state=C.RANDOM_STATE,
            eval_metric="logloss",
            scale_pos_weight=pos_weight,
            n_jobs=-1,
        ),
        {
            "n_estimators": [200, 400],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
        },
    ),
}
print(f"3 model tanımlandı. pos_weight (XGBoost) = {pos_weight:.3f}")

3 model tanımlandı. pos_weight (XGBoost) = 1.225


## 3. GridSearchCV — 5-fold

`scoring="recall"` — tıbbi vakada hasta kaçırmanın maliyeti yüksek olduğu için recall'a göre seç.
Final değerlendirmede zaten tüm metriklere bakacağız.

Çalışma süresi: küçük veri seti olduğu için ~30–60 saniye toplam.

In [4]:
results = {}
for name, (est, grid) in models.items():
    print(f"--- {name} GridSearch başlıyor ---")
    gs = GridSearchCV(
        estimator=est,
        param_grid=grid,
        scoring="recall",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    gs.fit(X_train, y_train)
    results[name] = gs
    print(f"  En iyi params: {gs.best_params_}")
    print(f"  CV recall (mean ± std): {gs.best_score_:.4f} ± "
          f"{gs.cv_results_['std_test_score'][gs.best_index_]:.4f}\n")

--- LogisticRegression GridSearch başlıyor ---
  En iyi params: {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}
  CV recall (mean ± std): 0.7933 ± 0.1070

--- RandomForest GridSearch başlıyor ---
  En iyi params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV recall (mean ± std): 0.7533 ± 0.1301

--- XGBoost GridSearch başlıyor ---
  En iyi params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
  CV recall (mean ± std): 0.7052 ± 0.0727



## 4. Confusion Matrix — her model için train + test

In [5]:
fig, axes = plt.subplots(len(results), 2, figsize=(10, 4 * len(results)))
for i, (name, gs) in enumerate(results.items()):
    best = gs.best_estimator_
    for j, (X_, y_, label) in enumerate([(X_train, y_train, "Train"),
                                          (X_test, y_test, "Test")]):
        y_pred = best.predict(X_)
        cm = confusion_matrix(y_, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=["Sağlıklı", "Hasta"])
        disp.plot(ax=axes[i, j], cmap="Blues", colorbar=False)
        axes[i, j].set_title(f"{name} — {label}")
fig.suptitle("Confusion Matrix — 3 model × 2 set", y=1.005, fontsize=14)
U.savefig(fig, C.FIG_DIR / "07_confusion_matrices.png")
print("Kaydedildi → outputs/figures/07_confusion_matrices.png")

Kaydedildi → outputs/figures/07_confusion_matrices.png


## 5. Metrik tablosu (Train / CV / Test)
Tıbbi vaka olduğu için tabloyu **Recall** sütununa göre sıralıyoruz.

In [6]:
def compute_metrics(model, X, y):
    pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
    }

rows = []
for name, gs in results.items():
    best = gs.best_estimator_
    train_m = compute_metrics(best, X_train, y_train)
    test_m = compute_metrics(best, X_test, y_test)
    cv_idx = gs.best_index_
    cv_results = gs.cv_results_

    rows.append({
        "model": name,
        "best_params": str(gs.best_params_),
        # train
        "train_accuracy": round(train_m["accuracy"], 4),
        "train_precision": round(train_m["precision"], 4),
        "train_recall": round(train_m["recall"], 4),
        "train_f1": round(train_m["f1"], 4),
        # CV (mean ± std)
        "cv_recall_mean": round(cv_results["mean_test_score"][cv_idx], 4),
        "cv_recall_std": round(cv_results["std_test_score"][cv_idx], 4),
        # test
        "test_accuracy": round(test_m["accuracy"], 4),
        "test_precision": round(test_m["precision"], 4),
        "test_recall": round(test_m["recall"], 4),
        "test_f1": round(test_m["f1"], 4),
    })

metrics_df = pd.DataFrame(rows).sort_values("test_recall", ascending=False).reset_index(drop=True)
U.save_table(metrics_df, C.METRICS_DIR / "model_results")
print("Kaydedildi → outputs/metrics/model_results.{csv,md}\n")
metrics_df

Kaydedildi → outputs/metrics/model_results.{csv,md}



,model,best_params,train_accuracy,train_precision,train_recall,train_f1,cv_recall_mean,cv_recall_std,test_accuracy,test_precision,test_recall,test_f1
0,LogisticRegression,"{'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}",0.7885,0.7547,0.7843,0.7692,0.7933,0.1070,0.8689,0.8125,0.9286,0.8667
1,RandomForest,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.9295,0.9388,0.9020,0.9200,0.7533,0.1301,0.8689,0.8333,0.8929,0.8621
2,XGBoost,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti...",0.9956,0.9903,1.0000,0.9951,0.7052,0.0727,0.8033,0.7857,0.7857,0.7857


## 6. En iyi modeli belirle ve hepsini kaydet

In [7]:
best_name = metrics_df.iloc[0]["model"]
print(f"En iyi (test recall'a göre): {best_name}")

for name, gs in results.items():
    joblib.dump(gs.best_estimator_, C.MODELS_DIR / f"{name}.joblib")
joblib.dump(best_name, C.MODELS_DIR / "best_model_name.joblib")

print("\nKaydedilen modeller:")
for f in sorted(C.MODELS_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

En iyi (test recall'a göre): LogisticRegression

Kaydedilen modeller:
  .gitkeep  (0.0 KB)
  best_model_name.joblib  (0.0 KB)
  imputation_medians.joblib  (0.2 KB)
  LogisticRegression.joblib  (1.3 KB)
  RandomForest.joblib  (1227.6 KB)
  scaler.joblib  (1.0 KB)
  selected_features.joblib  (0.1 KB)
  XGBoost.joblib  (321.5 KB)


## Çıktılar:

- `models/LogisticRegression.joblib`, `RandomForest.joblib`, `XGBoost.joblib`
- `models/best_model_name.joblib`
- `outputs/figures/07_confusion_matrices.png`
- `outputs/metrics/model_results.{csv,md}`